In [0]:
# ==============================================================================
# EXPLORAÇÃO E AUDITORIA DA CAMADA SILVER (DATABRICKS / SPARK)
# ==============================================================================
from pyspark.sql.functions import col, count, when, round

# 1. Carregamento das Tabelas do Unity Catalog
df_bronze = spark.table("workspace.default.bronze_listings")
df_silver = spark.table("workspace.default.silver_listings")

# ------------------------------------------------------------------------------
# A. AUDITORIA DE VOLUMETRIA E RETENÇÃO (BRONZE VS. SILVER)
# ------------------------------------------------------------------------------
total_bronze = df_bronze.count()
total_silver = df_silver.count()
linhas_eliminadas = total_bronze - total_silver
pct_retencao = (total_silver / total_bronze) * 100

cols_bronze = len(df_bronze.columns)
cols_silver = len(df_silver.columns)

print("=" * 60)
print("📊 RESUMO DA AUDITORIA DE VOLUMETRIA E ESTRUTURA")
print("=" * 60)
print(f" • Camada Bronze (Bruta)  : {total_bronze:,} linhas | {cols_bronze} colunas")
print(f" • Camada Silver (Tratada): {total_silver:,} linhas | {cols_silver} colunas")
print(f" • Linhas Descartadas    : {linhas_eliminadas:,}")
print(f" • Taxa de Retenção Base  : {pct_retencao:.2f}%")
print("=" * 60)

# ------------------------------------------------------------------------------
# B. ANÁLISE DE MOTIVOS DE DESCARTE NA BRONZE
# ------------------------------------------------------------------------------
nulos_coordenadas = df_bronze.filter(col("latitude").isNull() | col("longitude").isNull()).count()
nulos_bairro = df_bronze.filter(col("neighbourhood_cleansed").isNull()).count()
duplicados_id = total_bronze - df_bronze.dropDuplicates(["id"]).count()

print("\n🔍 DETALHAMENTO DAS ANOMALIAS FILTRADAS:")
print(f" - Registros sem Coordenadas Geográficas: {nulos_coordenadas}")
print(f" - Registros sem Bairro Definido       : {nulos_bairro}")
print(f" - IDs Duplicados Removidos            : {duplicados_id}")

# ------------------------------------------------------------------------------
# C. RELAÇÃO DE TODAS AS COLUNAS TRATADAS DA SILVER E SEUS TIPOS
# ------------------------------------------------------------------------------
print("\n📋 RELAÇÃO COMPLETA DAS COLUNAS SILVER E SUAS TIPAGENS:")
print("-" * 60)
for idx, (col_name, dtype) in enumerate(df_silver.dtypes, start=1):
    print(f"{idx:02d}. {col_name:<30} | Tipo: {dtype}")
print("-" * 60)

# ------------------------------------------------------------------------------
# D. VISUALIZAÇÃO INTERATIVA DAS PRIMEIRAS LINHAS DA SILVER
# ------------------------------------------------------------------------------
display(df_silver)